In [53]:
import pandas as pd
import numpy as np
import pickle

In [70]:
data_input_path = "../../data/final/raw_data/"

data_output_path = "../../data/final/keywords_and_location_names/"

In [55]:
# Load table with the original keywords with keyword category (downloaded from old github repo)
original_keyword_df = pd.read_csv(data_input_path + "keyword_list.csv")

original_keyword_df.rename(columns={"label":"keyword"}, inplace=True)

In [56]:
# Load with table with keywords translated by the Mashreq team
translation_keyword_df = pd.read_excel(data_input_path + "FS Predictions - Keywords - ENG - ARB.xlsx")

# 1. Merge Keyword file (because of keyword category) with the translation

In [57]:
# Join the two tables
keyword_df = translation_keyword_df.merge(original_keyword_df, left_on="English", right_on="keyword", how="left")

In [58]:
# Fill NAs in the the OLD keyword part of the joint table from above
keyword_df.iloc[:209] = keyword_df.iloc[:209].ffill()
keyword_df = keyword_df.drop(columns=["ID ", "Comment", "keyword"])

# Add the new keyword category "New Category" to the new keyword part of the joint table
keyword_df.loc[keyword_df["cluster"].isna(),"cluster"] = "New Category"

# Fill NAs in the NEW keyword part from above
keyword_df = keyword_df.ffill()

In [59]:
# Make sure that there are no leading or trailing whitespaces
keyword_df.loc[~keyword_df["English"].isna(), "English"] = keyword_df.loc[~keyword_df["English"].isna(), "English"].apply(lambda x: x.strip())
keyword_df.loc[~keyword_df["Arabic"].isna(), "Arabic"] = keyword_df.loc[~keyword_df["Arabic"].isna(), "Arabic"].apply(lambda x: x.strip())

In [60]:
keyword_categories = {'Conflicts and Violence': 'cv',
                    'Political Instability': 'pi',
                    'Humanitarian Aid': 'ha',
                    'Economic Issues': 'eci',
                    'Agricultural Production Issues': 'prs',
                    'Weather Shocks': 'ws',
                    'Food Crisis': 'fc',
                    'Land-related Issues': 'lri',
                    'Pests and Diseases': 'pad',
                    'Forced Displacement': 'fd',
                    'Environmental Issues': 'ei',
                    'Other': 'o',
                    'New Category': 'nc',
                    'Any Keyword': 'kw'}

In [61]:
# Make the group names upper case
keyword_df["cluster"] = keyword_df["cluster"].apply(lambda x: " ".join([word.capitalize() if word != "and" else 'and' for word in x.split(" ")]))

In [62]:
# Make sure all keywords are lower case
keyword_df["English"] = keyword_df["English"].str.lower()

In [63]:
# Create "ColumnName" column
keyword_df["ColumnName1"] = keyword_df["cluster"].apply(lambda x: keyword_categories[x] + "_")
keyword_df["ColumnName2"] = keyword_df["English"].apply(lambda x: "_".join(x.split(" ")))
keyword_df["ColumnName"] = keyword_df["ColumnName1"] + keyword_df["ColumnName2"]
keyword_df = keyword_df.drop(columns=["ColumnName1", "ColumnName2"])

In [65]:
# Delete duplicates that were created by adding the new category
keyword_df = keyword_df.loc[keyword_df["ColumnName"] != "nc_food_sovereignty"]
keyword_df = keyword_df.loc[keyword_df["ColumnName"] != "nc_failed_crops"]

In [66]:
# Repeat all keywords for the Any Keyword category
keyword_df_copy = keyword_df.copy()
keyword_df_copy["cluster"] = "Any Keyword"
keyword_df = pd.concat([keyword_df, keyword_df_copy]).reset_index(drop=True)

In [67]:
# Create entries for the category summaries
summary_columns_df = pd.DataFrame(data={"English": np.nan, "Arabic": np.nan, "ColumnName" : keyword_categories.values(), "cluster": keyword_categories.keys()})
keyword_df = pd.concat([keyword_df, summary_columns_df]).reset_index(drop=True)

In [68]:
# Rename the columns and sort the table
keyword_df.rename(columns={"English": "keyword_eng", "Arabic": "keyword_ara", "cluster": "group_name", "ColumnName": "column_name"}, inplace=True)
keyword_df.sort_values(by="column_name", inplace=True)
keyword_df.reset_index(drop=True, inplace=True)

In [71]:
# Save keyword table
keyword_df.to_csv(data_output_path + "keywords_dataframe.csv", index=False)

# 2. Create keyword dictionaries

In [74]:
# Load the keyword table
keyword_df = pd.read_csv(data_output_path + "keywords_dataframe.csv")

In [75]:
# Create dictionary for English Keywords (column_name: [keywords])
keyword_dict_eng = {}

for column_name in keyword_df.loc[~keyword_df["keyword_eng"].isna(), "column_name"].unique():
    keyword_dict_eng[column_name] = np.unique(keyword_df.loc[keyword_df["column_name"] == column_name, "keyword_eng"].values)

In [76]:
# Create dictionary for Arabic Keywords (column_name: [keywords])
keyword_dict_ara = {}

for column_name in keyword_df.loc[~keyword_df["keyword_ara"].isna(), "column_name"].unique():
    keyword_dict_ara[column_name] = np.unique(keyword_df.loc[keyword_df["column_name"] == column_name, "keyword_ara"].values)

In [77]:
# Check that both dictionaries have the same keys
assert len([key for key in keyword_dict_eng.keys() if key not in keyword_dict_ara.keys()]) == 0, "There are missing Arabic translations for some English keywords"
assert len([key for key in keyword_dict_ara.keys() if key not in keyword_dict_eng.keys()]) == 0, "There are missing English translations for some Arabic keywords"

In [79]:
# Save the English dictionary
file_path = data_output_path + 'id_english_keyword.pkl'

with open(file_path, 'wb') as f:
    pickle.dump(keyword_dict_eng, f)

In [84]:
# Save the Arabic dictionary
file_path = data_output_path + 'id_arabic_keyword.pkl'

with open(file_path, 'wb') as f:
    pickle.dump(keyword_dict_ara, f)